In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os

%matplotlib inline

# Kaplan–Meier Confidence Intervals for Droplet-Freezing Data

This notebook calculates fraction-frozen curves and **pointwise Kaplan–Meier confidence intervals** for one or more droplet-freezing experiments. It does not require temperature binning or an assumed distribution.

Optionally, it also calculates a cumulative ice-nucleation activity spectrum:

$$
K(T)=\frac{-\ln\left[1-F(T)\right]}
{\text{cumulative spectrum normalisation factor}}
$$

where \(F(T)\) is the fraction frozen at temperature \(T\). The normalisation factor may represent droplet volume, sample mass, surface area, or another appropriate quantity.

## Input and use

The CSV file should contain one column of freezing temperatures per experiment, with experiment names as headings. Blank and non-numeric cells are removed automatically.

1. Place the CSV file in the notebook folder.
2. Enter the filenames and confidence level in the **Setup** cell.
3. Set `CALCULATE_CUMULATIVE_SPECTRUM` to `True` or `False`.
4. If required, enter the cumulative spectrum normalisation factor.
5. Run all cells in order.

The shaded areas are pointwise confidence intervals, not confidence bands for the whole curve. The notebook assumes that every value is an observed freezing event and does not currently handle censored observations. curves and confidence intervals.
d confidence intervals.


In [ ]:
# ============================================================
# Input and output files and Setup
# ============================================================

INPUT_FILENAME = "example_data_KM_intervals.csv"
OUTPUT_FILENAME = "example_data_KM_intervals.csv"
FIGURE_FILENAME = "example_data_KM_intervals.png"

# Use 1.96 for 95% confidence intervals
# Use 1.64 for 90% confidence intervals
Z_SCORE = 1.96

FIGURE_DPI = 300

# Calculate and plot the cumulative spectrum
CALCULATE_CUMULATIVE_SPECTRUM = True

# Cumulative spectrum normalisation factor
# Examples:
# Volume of droplet in litres: produces K(T) in L^-1
# Mass in grams per droplet: produces n_m in g^-1
# Surface area per droplet in cm^2: produces n_s in cm^-2

CUMULATIVE_SPECTRUM_NORMALISATION_FACTOR = 1

K_FIGURE_FILENAME = "Daily_data_K_intervals.png"


In [ ]:
# ============================================================
# Confidence-interval function
# ============================================================

def survival_cis(S, N, z=1.96):

    with np.errstate(divide="ignore", invalid="ignore"):
        log_log_se = np.sqrt(
            (1 - S) / (N * S * np.log(S) ** 2))

    exponent_term = z * log_log_se

    return (
        S ** np.exp(exponent_term),
        S ** np.exp(-exponent_term))

In [ ]:
# ============================================================
# Kaplan–Meier calculation
# ============================================================

def calculate_km_loglog(series, group_name):

    # Clean and sort
    series = pd.to_numeric(series, errors="coerce")

    temps = (
        series
        .dropna()
        .sort_values(ascending=False)
        .reset_index(drop=True))

    N_total = len(temps)

    if N_total == 0:
        print(f"Warning: '{group_name}' contains no numeric data.")
        return None

    # Group observations by temperature
    km_data = (
        temps
        .value_counts()
        .sort_index(ascending=False)
        .reset_index())

    km_data.columns = ["tf", "events"]

    # Number at risk
    km_data["cumulative_events_before"] = (
        km_data["events"]
        .cumsum()
        .shift(1, fill_value=0))

    km_data["at_risk"] = (
        N_total - km_data["cumulative_events_before"]
    )

    # Fraction frozen
    km_data["cumulative_events"] = (
        km_data["events"].cumsum())

    km_data["fraction_frozen"] = (
        km_data["cumulative_events"] / N_total)

    # Kaplan–Meier survival
    km_data["term"] = (
        1 - km_data["events"] / km_data["at_risk"])

    km_data["S_T"] = km_data["term"].cumprod()

    # Log–log confidence intervals
    km_data["CI_lower"], km_data["CI_upper"] = survival_cis(
        km_data["S_T"],
        N_total,
        z=Z_SCORE)

    # Convert survival intervals to fraction-frozen intervals
    km_data["FF_CI_lower"] = 1 - km_data["CI_upper"]
    km_data["FF_CI_upper"] = 1 - km_data["CI_lower"]

    # Fill the final interval where S(T) = 0
    if len(km_data) >= 2 and km_data["S_T"].iloc[-1] == 0:

        km_data.loc[
            km_data.index[-1],
            "FF_CI_lower"
        ] = km_data["FF_CI_lower"].iloc[-2]

        km_data.loc[
            km_data.index[-1],
            "FF_CI_upper"] = 1.0

    # Add group name and total number
    km_data["Group"] = group_name
    km_data["N_total"] = N_total

    columns = [
        "Group",
        "N_total",
        "tf",
        "at_risk",
        "events",
        "fraction_frozen",
        "S_T",
        "FF_CI_lower",
        "FF_CI_upper"]

    return km_data[columns]

In [ ]:

# ============================================================
# Read and process the CSV
# ============================================================
print(f"Reading from: {INPUT_FILENAME}")

df = pd.read_csv(INPUT_FILENAME)

# Remove spaces around column headings
df.columns = df.columns.str.strip()

# Treat every column as a separate group
groups = df.columns.tolist()

print(f"Found {len(groups)} groups: {groups}")

all_results = []
valid_groups = []

for group in groups:

    result = calculate_km_loglog(
        df[group],
        group)

    if result is not None:
        all_results.append(result)
        valid_groups.append(group)

final_df = pd.concat(
    all_results,
    ignore_index=True)

print("Data processing complete.")

display(final_df.head())

In [ ]:
# ============================================================
# Calculate the cumulative spectrum, K(T)
# ============================================================

if CALCULATE_CUMULATIVE_SPECTRUM:

    with np.errstate(divide="ignore", invalid="ignore"):

        final_df["K"] = (
            -np.log(1 - final_df["fraction_frozen"])
            / CUMULATIVE_SPECTRUM_NORMALISATION_FACTOR
        )

        final_df["K_CI_lower"] = (
            -np.log(1 - final_df["FF_CI_lower"])
            / CUMULATIVE_SPECTRUM_NORMALISATION_FACTOR
        )

        final_df["K_CI_upper"] = (
            -np.log(1 - final_df["FF_CI_upper"])
            / CUMULATIVE_SPECTRUM_NORMALISATION_FACTOR
        )

    print("K(T) and its confidence intervals calculated.")

else:

    print("K(T) calculation switched off.")

In [ ]:
# ============================================================
# Save the results
# ============================================================

final_df.to_csv(
    OUTPUT_FILENAME,
    index=False)

print(f"Results saved to: {OUTPUT_FILENAME}")
print(f"Files saved in: {os.getcwd()}")

In [ ]:
# ============================================================
# Plot and save the figure
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=100)

colors = cm.viridis(
    np.linspace(0, 0.9, len(valid_groups)))

for i, group in enumerate(valid_groups):

    subset = final_df[
        final_df["Group"] == group]

    color = colors[i]

    ax.step(
        subset["tf"],
        subset["fraction_frozen"],
        where="post",
        label=group,
        color=color,
        linewidth=2)

    ax.fill_between(
        subset["tf"],
        subset["FF_CI_lower"],
        subset["FF_CI_upper"],
        step="post",
        color=color,
        alpha=0.15)

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Fraction frozen")
ax.set_title("Fraction-frozen curves")
ax.set_ylim(0, 1)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()

fig.tight_layout()

fig.savefig(
    FIGURE_FILENAME,
    dpi=FIGURE_DPI,
    bbox_inches="tight")

print(f"Figure saved to: {FIGURE_FILENAME}")

plt.show()

In [ ]:
# ============================================================
# Plot K(T) and its confidence intervals
# ============================================================

if CALCULATE_CUMULATIVE_SPECTRUM:

    fig, ax = plt.subplots(
        figsize=(10, 6),
        dpi=100
    )

    colors = cm.viridis(
        np.linspace(0, 0.9, len(valid_groups))
    )

    for i, group in enumerate(valid_groups):

        subset = final_df[
            final_df["Group"] == group
        ].copy()

        color = colors[i]

        # Remove undefined or infinite values
        finite = (
            np.isfinite(subset["K"])
            & np.isfinite(subset["K_CI_lower"])
            & np.isfinite(subset["K_CI_upper"])
        )

        subset = subset[finite]

        ax.step(
            subset["tf"],
            subset["K"],
            where="post",
            label=group,
            color=color,
            linewidth=2
        )

        ax.fill_between(
            subset["tf"],
            subset["K_CI_lower"],
            subset["K_CI_upper"],
            step="post",
            color=color,
            alpha=0.15
        )

    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel(r"$K(T)$")
    ax.set_title("Cumulative ice-nucleation activity spectrum")
    ax.set_yscale("log")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend()

    fig.tight_layout()

    fig.savefig(
        K_FIGURE_FILENAME,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    print(f"K(T) figure saved to: {K_FIGURE_FILENAME}")

    plt.show()